In [1]:
import pandas as pd
import os
from pathlib import Path

In [2]:
print(os.getcwd())
project_root = Path.cwd().parent

/home/dread/Documents/Personal projects/SA-Water-Dam-Level-Predictor/notebooks


In [3]:
daily_rainfall = "../data/processed/daily_rainfall.csv"
clean_dam = "../data/processed/clean_dam.csv"

In [4]:
clean_dam_df = pd.read_csv(clean_dam)
daily_rainfall_df = pd.read_csv(daily_rainfall)
print(daily_rainfall_df.info())
print(f"daily rainfall shape:{daily_rainfall_df.shape}")
print(clean_dam_df.info())
print(f"daily rainfall shape:{daily_rainfall_df.shape}")

<class 'pandas.DataFrame'>
RangeIndex: 9617 entries, 0 to 9616
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            9617 non-null   str    
 1   water_level_mm  9617 non-null   float64
 2   year            9617 non-null   int64  
 3   month           9617 non-null   int64  
dtypes: float64(1), int64(2), str(1)
memory usage: 300.7 KB
None
daily rainfall shape:(9617, 4)
<class 'pandas.DataFrame'>
RangeIndex: 186 entries, 0 to 185
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   willDam     186 non-null    str    
 1   River       186 non-null    str    
 2   Indicators  186 non-null    str    
 3   FSC         186 non-null    float64
 4   This Week   186 non-null    float64
 5   Last Week   186 non-null    str    
 6   Last Year   186 non-null    str    
dtypes: float64(2), str(5)
memory usage: 10.3 KB
None
daily rainfall shape:(9

In [5]:
###These are the rollling lag features being created: lag being 1, 7, 14, 28, 56 and 84###

daily_rainfall_df["lag_1"] = daily_rainfall_df["water_level_mm"].shift(1)
daily_rainfall_df["lag_7"] = daily_rainfall_df["water_level_mm"].shift(7)
daily_rainfall_df["lag_14"] = daily_rainfall_df["water_level_mm"].shift(14)
daily_rainfall_df["lag_28"] = daily_rainfall_df["water_level_mm"].shift(28)
daily_rainfall_df["lag_56"] = daily_rainfall_df["water_level_mm"].shift(56)
daily_rainfall_df["lag_84"] = daily_rainfall_df["water_level_mm"].shift(84)

print(daily_rainfall_df.info())
print(f"daily rainfall shape:{daily_rainfall_df.shape}")
print(f"lag_values\n:{daily_rainfall_df[["lag_7", "lag_14", "lag_84"]].tail(10)}")

<class 'pandas.DataFrame'>
RangeIndex: 9617 entries, 0 to 9616
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            9617 non-null   str    
 1   water_level_mm  9617 non-null   float64
 2   year            9617 non-null   int64  
 3   month           9617 non-null   int64  
 4   lag_1           9616 non-null   float64
 5   lag_7           9610 non-null   float64
 6   lag_14          9603 non-null   float64
 7   lag_28          9589 non-null   float64
 8   lag_56          9561 non-null   float64
 9   lag_84          9533 non-null   float64
dtypes: float64(7), int64(2), str(1)
memory usage: 751.5 KB
None
daily rainfall shape:(9617, 10)
lag_values
:         lag_7     lag_14    lag_84
9607  0.847092   0.311230  1.278165
9608  2.786286   2.848348  1.955726
9609  2.604120   0.321911  4.560972
9610  7.049541   1.401302  2.280130
9611  2.314826  12.050176  3.267598
9612  3.228934   1.274137  2.405660
961

In [6]:
"""This is the rolling window feaatures"""
daily_rainfall_df["rolling_mean_28"] = daily_rainfall_df["water_level_mm"].rolling(28).mean()
daily_rainfall_df["rolling_mean_84"] = daily_rainfall_df["water_level_mm"].rolling(84).mean()
daily_rainfall_df["weekly_sum"] = daily_rainfall_df["water_level_mm"].rolling(7).sum()
daily_rainfall_df["lag_7_weekly_sum"] = daily_rainfall_df["weekly_sum"].shift(7)
daily_rainfall_df["week_over_week_pct"] = (daily_rainfall_df["weekly_sum"]-daily_rainfall_df["lag_7_weekly_sum"])/daily_rainfall_df["lag_7_weekly_sum"] * 100
daily_rainfall_df["week_over_week_pct"] = daily_rainfall_df["week_over_week_pct"].fillna(0)

In [7]:
print(daily_rainfall_df["week_over_week_pct"])
print(daily_rainfall_df["water_level_mm"].describe())

0        0.000000
1        0.000000
2        0.000000
3        0.000000
4        0.000000
          ...    
9612    -2.077312
9613    -4.257066
9614   -41.844659
9615   -71.176527
9616   -64.622910
Name: week_over_week_pct, Length: 9617, dtype: float64
count    9617.000000
mean        1.149207
std         1.597756
min         0.000000
25%         0.080735
50%         0.448099
75%         1.633875
max        19.466718
Name: water_level_mm, dtype: float64


In [8]:
"""Significant rainy days above 75%"""
percent_75 = 1.633875
daily_rainfall_df["significant_rain_days"] = daily_rainfall_df["water_level_mm"]>percent_75



In [9]:
"""days_since_significant_rain count"""
count = 0
true_count_lst = []
for values in daily_rainfall_df["significant_rain_days"]:
    if values == False:
        count += 1
    else:
        count = 0
    true_count_lst.append(count)
daily_rainfall_df["days_since_significant_rain"] = true_count_lst
print(daily_rainfall_df["days_since_significant_rain"].head(15))


0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     1
9     2
10    3
11    4
12    5
13    6
14    0
Name: days_since_significant_rain, dtype: int64


In [10]:
"""Formatting date data to datetime data"""
daily_rainfall_df["date"] = pd.to_datetime(daily_rainfall_df["date"], errors="coerce", format='%Y-%m-%d')
print(daily_rainfall_df["date"])

0      2000-01-01
1      2000-01-02
2      2000-01-03
3      2000-01-04
4      2000-01-05
          ...    
9612   2026-04-26
9613   2026-04-27
9614   2026-04-28
9615   2026-04-29
9616   2026-04-30
Name: date, Length: 9617, dtype: datetime64[us]


In [11]:
daily_rainfall_df["quarter"] = daily_rainfall_df["date"].dt.quarter
daily_rainfall_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9617 entries, 0 to 9616
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   date                         9617 non-null   datetime64[us]
 1   water_level_mm               9617 non-null   float64       
 2   year                         9617 non-null   int64         
 3   month                        9617 non-null   int64         
 4   lag_1                        9616 non-null   float64       
 5   lag_7                        9610 non-null   float64       
 6   lag_14                       9603 non-null   float64       
 7   lag_28                       9589 non-null   float64       
 8   lag_56                       9561 non-null   float64       
 9   lag_84                       9533 non-null   float64       
 10  rolling_mean_28              9590 non-null   float64       
 11  rolling_mean_84              9534 non-null   float64  

In [12]:
print(daily_rainfall_df["month"].head())

0    1
1    1
2    1
3    1
4    1
Name: month, dtype: int64


In [13]:
daily_rainfall_df["wet_season"] = daily_rainfall_df["month"][(daily_rainfall_df["month"] < 5) | (daily_rainfall_df["month"] > 10)]
daily_rainfall_df["wet_dry_flag"] = daily_rainfall_df["month"].isin(daily_rainfall_df["wet_season"])
print(daily_rainfall_df["wet_dry_flag"].count())
print(daily_rainfall_df["wet_dry_flag"].head())

9617
0    True
1    True
2    True
3    True
4    True
Name: wet_dry_flag, dtype: bool


In [14]:
"""This is what tells me what year the season started, this is beinng used for the days into season feature engineering."""
daily_rainfall_df["season_start_year"] = daily_rainfall_df.apply(
    lambda row: row["date"].year if row["date"].month >= 10 else row["date"].year -1,
    axis = 1
)

In [15]:
daily_rainfall_df["days_into_season"] = (
    daily_rainfall_df.groupby("season_start_year")["date"].transform(lambda x: (x - x.min()).dt.days)
    )
print(daily_rainfall_df["days_into_season"].head())
print(daily_rainfall_df.info())

0    0
1    1
2    2
3    3
4    4
Name: days_into_season, dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 9617 entries, 0 to 9616
Data columns (total 22 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   date                         9617 non-null   datetime64[us]
 1   water_level_mm               9617 non-null   float64       
 2   year                         9617 non-null   int64         
 3   month                        9617 non-null   int64         
 4   lag_1                        9616 non-null   float64       
 5   lag_7                        9610 non-null   float64       
 6   lag_14                       9603 non-null   float64       
 7   lag_28                       9589 non-null   float64       
 8   lag_56                       9561 non-null   float64       
 9   lag_84                       9533 non-null   float64       
 10  rolling_mean_28              9590 non-null   fl

In [16]:
daily_rainfall_df["target_30"] = daily_rainfall_df["water_level_mm"].rolling(30).sum().shift(-30)
daily_rainfall_df["target_60"] = daily_rainfall_df["water_level_mm"].rolling(60).sum().shift(-60)
daily_rainfall_df["target_90"] = daily_rainfall_df["water_level_mm"].rolling(90).sum().shift(-90)
daily_rainfall_df["lag_1_target_30"] = daily_rainfall_df['target_30'].shift(1)
print(daily_rainfall_df[["target_30", "target_60", "target_90"]].head())


   target_30   target_60   target_90
0  71.443829  185.376647  267.337057
1  65.782134  184.194848  262.486257
2  64.834308  184.226363  263.810506
3  63.353634  184.478251  264.650207
4  58.524810  179.249388  260.703346


In [17]:
daily_rainfall_df = daily_rainfall_df.dropna()
# print(daily_rainfall_df.isnull().sum())
print(daily_rainfall_df.head())

         date  water_level_mm  year  month     lag_1     lag_7    lag_14  \
84 2000-03-25        2.667252  2000      3  7.038533  5.152590  0.866577   
85 2000-03-26        5.065514  2000      3  2.667252  1.081687  1.942799   
86 2000-03-27        3.869131  2000      3  5.065514  3.774888  1.212185   
87 2000-03-28        2.482193  2000      3  3.869131  1.087285  1.506839   
88 2000-03-29        3.010641  2000      3  2.482193  1.344326  2.519668   

      lag_28    lag_56    lag_84  ...  days_since_significant_rain  quarter  \
84  0.713604  1.256391  9.107801  ...                            0        1   
85  0.627804  0.068120  5.781268  ...                            0        1   
86  0.799240  0.054767  1.829304  ...                            0        1   
87  1.622073  0.119573  2.341147  ...                            0        1   
88  3.079031  0.881478  5.645532  ...                            0        1   

    wet_season  wet_dry_flag  season_start_year  days_into_season  t

In [18]:
daily_rainfall_df.to_csv("../data/feature_engineered/engineered_data.csv", index=False)